# STEP 2 - Tokenizer family determines vulnerability

Step 1 returned 0% evasion because **BanglaBERT's tokenizer deletes the attack characters
before the model ever sees them.** `BertTokenizer._clean_text()` drops every character whose
Unicode category starts with `C`; ZWNJ/ZWJ/ZWSP are all category **Cf**.

That is not a failure - it is the finding. BERT-family models are *accidentally* immune.
SentencePiece/BPE models are not. Verified locally:

| model | tokenizer | tokens change under attack? |
|---|---|---|
| csebuetnlp/banglabert | BertTokenizer | No - immune |
| google/muril-base-cased | BertTokenizer | No - immune |
| xlm-roberta-base | XLMRobertaTokenizer (SentencePiece) | **Yes - vulnerable** |

This notebook proves it behaviourally. Same Kaggle settings as Step 1
(**GPU T4 x2, Internet ON, dataset `naurosromim/bdshs`**). ~45 min.


In [ ]:
# 1. Setup (same as Step 1)
!pip install -q "emoji==1.7.0" regex ftfy
!pip install -q git+https://github.com/csebuetnlp/normalizer

import os, glob, json, random, unicodedata, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV="cuda" if torch.cuda.is_available() else "cpu"
ZWNJ,ZWJ,ZWSP="\u200c","\u200d","\u200b"; HASANT="\u09cd"
print("device:",DEV)

In [ ]:
# 2. Attack lib
def is_bangla(c): return 0x0980<=ord(c)<=0x09FF
def inert_positions(t):
    out=[]
    for i in range(1,len(t)):
        a,b=t[i-1],t[i]
        if not(is_bangla(a) and is_bangla(b)): continue
        if unicodedata.category(a)!="Lo" or unicodedata.category(b)!="Lo": continue
        out.append(i)
    return out
def attack(t,budget=1,char=ZWNJ,seed=0):
    p=inert_positions(t)
    if not p: return t
    r=random.Random(seed)
    for i in sorted(r.sample(p,min(budget,len(p))),reverse=True): t=t[:i]+char+t[i:]
    return t
def defend_position_aware(t):
    o=[]
    for i,ch in enumerate(t):
        if ch in (ZWNJ,ZWJ):
            pv=t[i-1] if i>0 else ""; nx=t[i+1] if i+1<len(t) else ""
            if pv==HASANT or nx==HASANT: o.append(ch)
            continue
        if ch==ZWSP: continue
        o.append(ch)
    return "".join(o)

In [ ]:
# 3. TOKENIZER VULNERABILITY SURVEY  (no training - seconds, big table for the paper)
CANDIDATES=[
 "csebuetnlp/banglabert","csebuetnlp/banglishbert","sagorsarker/bangla-bert-base",
 "google/muril-base-cased","bert-base-multilingual-cased",
 "xlm-roberta-base","ai4bharat/indic-bert","csebuetnlp/banglat5",
 "google/mt5-base","facebook/mbart-large-50",
]
probe="\u09a4\u09c1\u0987 \u098f\u0995\u099f\u09be \u09ac\u09be\u099c\u09c7 \u09b2\u09cb\u0995"
rows=[]
for name in CANDIDATES:
    try: tk=AutoTokenizer.from_pretrained(name)
    except Exception as e:
        rows.append({"model":name,"tokenizer":"LOAD_FAIL","vulnerable":None,
                     "clean_len":None,"atk_len":None}); continue
    atk=attack(probe,budget=2,char=ZWNJ,seed=1)
    a,b=tk.tokenize(probe),tk.tokenize(atk)
    rows.append({"model":name,"tokenizer":type(tk).__name__,
                 "vulnerable":a!=b,"clean_len":len(a),"atk_len":len(b)})
    print(f"{name:34} {type(tk).__name__:26} vulnerable={a!=b}")
tok_survey=pd.DataFrame(rows)
tok_survey.to_csv("/kaggle/working/tokenizer_survey.csv",index=False)
print(); print(tok_survey.to_string(index=False))

In [ ]:
# 4. Data
paths=glob.glob("/kaggle/input/**/*.csv",recursive=True)
pick=lambda k:[p for p in paths if k in os.path.basename(p).lower()][0]
train=pd.read_csv(pick("train")); test=pd.read_csv(pick("test"))
obj=[c for c in train.columns if train[c].dtype==object]
TEXT=max(obj,key=lambda c:train[c].astype(str).str.len().mean())
LABEL=[c for c in train.columns if c!=TEXT and train[c].dropna().nunique()==2][0]
def prep(df):
    d=df[[TEXT,LABEL]].dropna().copy(); d.columns=["text","label"]
    d["text"]=d["text"].astype(str); d["label"]=d["label"].astype(int)
    return d[d["text"].str.strip().str.len()>0].reset_index(drop=True)
train,test=prep(train),prep(test)

N_TRAIN=20000        # raise to len(train) for the FINAL paper runs
train=train.sample(min(N_TRAIN,len(train)),random_state=SEED).reset_index(drop=True)
HATE=1
print("train",len(train),"test",len(test))

In [ ]:
# 5. Train + attack one model
MAXLEN=128; EPOCHS=2; BS=32

def run(model_name):
    print("\n"+"="*60); print(model_name); print("="*60)
    tk=AutoTokenizer.from_pretrained(model_name)
    mdl=AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2).to(DEV)
    enc=lambda ts: tk([normalize(t) for t in ts],truncation=True,padding="max_length",
                      max_length=MAXLEN,return_tensors="pt")
    e=enc(train["text"].tolist()); y=torch.tensor(train["label"].values)
    dl=DataLoader(TensorDataset(e["input_ids"],e["attention_mask"],y),batch_size=BS,shuffle=True)
    opt=torch.optim.AdamW(mdl.parameters(),lr=2e-5)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=2e-5,total_steps=EPOCHS*len(dl),pct_start=0.1)
    sc=torch.cuda.amp.GradScaler(); mdl.train()
    for ep in range(EPOCHS):
        for st,(i_,m_,l_) in enumerate(dl):
            i_,m_,l_=i_.to(DEV),m_.to(DEV),l_.to(DEV); opt.zero_grad()
            with torch.cuda.amp.autocast(): loss=mdl(input_ids=i_,attention_mask=m_,labels=l_).loss
            sc.scale(loss).backward(); sc.step(opt); sc.update(); sch.step()
            if st%200==0: print(f"  ep{ep} {st}/{len(dl)} loss {loss.item():.4f}")
    mdl.eval()

    @torch.no_grad()
    def pred(ts,bs=128):
        o=[]
        for i in range(0,len(ts),bs):
            b=enc(ts[i:i+bs])
            with torch.cuda.amp.autocast():
                lg=mdl(input_ids=b["input_ids"].to(DEV),attention_mask=b["attention_mask"].to(DEV)).logits
            o.append(lg.float().argmax(-1).cpu())
        return torch.cat(o).numpy()

    tt=test["text"].tolist(); ty=test["label"].values
    cp=pred(tt); acc=accuracy_score(ty,cp); f1=f1_score(ty,cp)
    hit=np.where((ty==HATE)&(cp==HATE))[0][:1500]
    sub=[tt[i] for i in hit]
    print(f"clean acc {acc:.4f} f1 {f1:.4f} | targets {len(sub)}")

    out=[]
    for cn,ch in [("ZWNJ",ZWNJ),("ZWJ",ZWJ),("ZWSP",ZWSP)]:
        for bud in [1,3,5]:
            atk=[attack(t,budget=bud,char=ch,seed=SEED) for t in sub]
            ev=float((pred(atk)!=HATE).mean())
            # does our defense restore detection?
            dfd=float((pred([defend_position_aware(a) for a in atk])!=HATE).mean())
            out.append({"model":model_name,"tokenizer":type(tk).__name__,"char":cn,
                        "budget":bud,"evasion_rate":round(ev,4),
                        "evasion_after_our_defense":round(dfd,4),
                        "clean_acc":round(acc,4),"clean_f1":round(f1,4)})
            print(f"  {cn:5} b={bud} evasion {ev:.2%} -> after defense {dfd:.2%}")
    del mdl; torch.cuda.empty_cache()
    return out

In [ ]:
# 6. Run all three  (~45 min)
MODELS=["csebuetnlp/banglabert","xlm-roberta-base","google/muril-base-cased"]
allres=[]
for m in MODELS:
    try: allres+=run(m)
    except Exception as ex: print("FAILED",m,type(ex).__name__,ex)
res=pd.DataFrame(allres); res.to_csv("/kaggle/working/step2_evasion.csv",index=False)
print(res.to_string(index=False))

In [ ]:
# 7. In-the-wild zero-width usage - what ARE those 908 inert occurrences?
allt=pd.concat([train["text"],test["text"]]).tolist()
ex=[]
for t in allt:
    for i,c in enumerate(t):
        if c in (ZWNJ,ZWJ):
            pv=t[i-1] if i>0 else ""; nx=t[i+1] if i+1<len(t) else ""
            ex.append({"kind":"functional" if (pv==HASANT or nx==HASANT) else "inert",
                       "char":"ZWNJ" if c==ZWNJ else "ZWJ",
                       "ctx":t[max(0,i-12):i+12].replace(ZWNJ,"<ZWNJ>").replace(ZWJ,"<ZWJ>")})
exdf=pd.DataFrame(ex); exdf.to_csv("/kaggle/working/wild_zw_examples.csv",index=False)
print(exdf["kind"].value_counts().to_string())


In [ ]:
# 8. SUMMARY  <-- copy this whole block back
print("="*64); print("STEP 2 SUMMARY"); print("="*64)
print("\n-- tokenizer vulnerability survey --")
print(tok_survey.to_string(index=False))
print("\n-- behavioural evasion by model --")
print(res.to_string(index=False))
print("\n-- in-the-wild zero-width --")
print(exdf["kind"].value_counts().to_string())
print("="*64)